# Predicting Race Outcomes using Machine Learning Models

Name: Lily Nguyen

UT Eid: lmn934

In [1747]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

### F1 Quick Summary

Formula 1 is the highest level of international motorsport, featuring 10 teams (aka "constructors") and a total of 20 drivers, with each team having 2 drivers. Each race weekend consists of practice sessions, a qualifying session that determines starting order, and the main race. Drivers earn points based on their finishing position, with 25 points awarded for 1st place, 18 for 2nd, 15 for 3rd, continuing down to 1 point for 10th place. An additional point is awarded for the driver who sets the fastest lap if they finish in the top 10. Throughout the season, which typically includes 20-24 races held arouund the world, points are accumulated toward both the Driver's Championship (awarded to the driver with most points) and the Constructor's Championship (awarded to the team with the highest combined points from its two drivers).


### Public Datasets

We plan to use multiple csv files from a public Formula 1 Dataset page on Kaggle and then merge them together to use with our ML models. The page includes data from the years 1950-present.

https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020/data
1. **results.csv** : contains race results for each driver including finishing position, points scored, and associated driver/constructor IDs
2. **qualifying.csv** : records qualifying poisitions and times for each driver
3. **constructors.csv** : provides constructor (team) information, will use one-hot encoding later to convert categorical ata to machine-understandable information (binaries)
4. **pit_stops.csv** : lists pit stop events for each driver during a race
5. **driver_standings.csv** : includes cumulative season points for each driver before a race, which provide context for performance differences accross tracks and seasons
6. **races.csv** : includes info on the races themselves such as 'year' and 'circuitId', which can give context for differences across tracks and seasons

## Research Question

**<u>Is it possible to predict whether a Formula 1 driver will score points (finish in the top 10) based on factors such as qualifying position, pit stops, team, and season performance?</u>**

This is a binary classification problem, where a target of 1 means the driver scored points, and target of 0 means the driver didn't score any points. 

## Defining the Project

The dataset contains historical Formula 1 racing data consiting of races, drivers,constructors, qualifying session, pit stops, and season standings. We merged these datasets into a single dataframe where each row represents an individual driver's performance in a race along with features such as qualifying position, pit stop data, season points, and team information.

The dataset originally included data from the years 1950-present. To ensure data quality, we filtered races from 2010 onwards since older seasons lacked some of the qualifyings data. After merging and preprocessing, we randomly reduced the dataset to 2000 driver-race records to fit with the assignment guidelines. Missing values such as pit stop durations and qualifying positions were handled as shown below.


#### Expectations for the results

We expect the qualifying position to me the most important factor in predicting whether a driver scores points, since starting closer to the front strongly correlates with finishing with the top racers. Season points could also be a strong precictor, since drivers who have performed well throughout the season typically continue to do so in succeeding races, on average. The team (constructor) and pit stop strategy may also influence the results, but these may be less significant than the ones mentioned previously. We hope to achieve a moderately-high accuracy using our model(s), so roughly around the 70-85% range.

#### Plan for evaluation

We plan to evaluate this project using a train/test split of 80/20. The main metric we plan to evaluate our models with is accuracy, which measures the overall percentage of correct predictions. Additionally, we plan to use the precision, recall, and f1-score from the classification_report to better understand the model's predictive ability for the two classes and evaluate for balanced performance. We also plan to evaluate the feature importance attribute from the random forest model to see whih factors contributed the most to predictions.

### Implementation

In [1748]:
# Load in datasets
results = pd.read_csv('results.csv')
qualifying = pd.read_csv('qualifying.csv')
constructors = pd.read_csv('constructors.csv')
pit_stops = pd.read_csv('pit_stops.csv')
driver_standings = pd.read_csv('driver_standings.csv')
races = pd.read_csv('races.csv')

# Get column names
print('\nqualifying.csv\n', qualifying.columns)
print('\ndrivers\n', drivers.columns)
print('\nconstructors\n', constructors.columns)
print('\npit_stops\n', pit_stops.columns)
print('\ndriver_standings\n', driver_standings.columns)
print('\nraces\n', races.columns)


qualifying.csv
 Index(['qualifyId', 'raceId', 'driverId', 'constructorId', 'number',
       'position', 'q1', 'q2', 'q3'],
      dtype='object')

drivers
 Index(['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob',
       'nationality', 'url'],
      dtype='object')

constructors
 Index(['constructorId', 'constructorRef', 'name', 'nationality', 'url'], dtype='object')

pit_stops
 Index(['raceId', 'driverId', 'stop', 'lap', 'time', 'duration',
       'milliseconds'],
      dtype='object')

driver_standings
 Index(['driverStandingsId', 'raceId', 'driverId', 'points', 'position',
       'positionText', 'wins'],
      dtype='object')

races
 Index(['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url',
       'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time',
       'quali_date', 'quali_time', 'sprint_date', 'sprint_time'],
      dtype='object')


In [1749]:
# Add in column for points scored binary to use as target variable
# points are scored if racers finish in the top 10 of each grand prix event
results['scored_points'] = (results['positionOrder'] <= 10).astype(int)
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,scored_points
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1,1


In [1750]:
# Merge the qualifying data onto results data using raceId and driverId
qualifying.rename(columns={'position': 'qualifying_position'}, inplace=True)
df = results.merge(qualifying[['raceId', 'driverId', 'qualifying_position']], on=['raceId', 'driverId'], how='left')

# Merge constructor data
df = df.merge(constructors[['constructorId', 'name']], on=['constructorId'], how='left')
df.rename(columns={'name': 'team_name'}, inplace=True)

# Merge pit_stop data
pit_stops['milliseconds'] = pit_stops['milliseconds'].astype(int)
pitstop_features = pit_stops.groupby(['raceId', 'driverId']).agg(pit_stop_count=('stop', 'count'), avg_pit_duration=('milliseconds', 'mean')).reset_index()
df = df.merge(pitstop_features, on=['raceId', 'driverId'], how='left')

# fill in missing pit stop data
df['pit_stop_count'] = df['pit_stop_count'].fillna(0) # no data
df['avg_pit_duration'] = df['avg_pit_duration'].fillna(0)

# Merge driver_standings (season points prior to current race)
driver_standings.rename(columns={'points': 'season_points'}, inplace=True)
df = df.merge(driver_standings[['raceId', 'driverId', 'season_points']], on=['raceId', 'driverId'], how='left')
df

# merge races data
# 'year': filter for more recent seasons, let model learn from recent performance trends
# 'circuitId': different tracks favor different teams and drivers (eg. high downforce tracks like Monaco favor certain teams)
df = df.merge(races[['raceId', 'year', 'circuitId']], on='raceId', how='left')

# Drop unecessary columns, keep only the useful ones
df = df[['raceId', 'driverId', 'constructorId', 'team_name', 'qualifying_position', 'pit_stop_count', 'avg_pit_duration', 'season_points', 'year', 'circuitId', 'scored_points']]
df.head()

,raceId,driverId,constructorId,team_name,qualifying_position,pit_stop_count,avg_pit_duration,season_points,year,circuitId,scored_points
0,18,1,1,McLaren,1.0,0.0,0.0,10.0,2008,1,1
1,18,2,2,BMW Sauber,5.0,0.0,0.0,8.0,2008,1,1
2,18,3,3,Williams,7.0,0.0,0.0,6.0,2008,1,1
3,18,4,4,Renault,12.0,0.0,0.0,5.0,2008,1,1
4,18,5,1,McLaren,3.0,0.0,0.0,4.0,2008,1,1


In [1751]:
# handle remaining NaN values
df.isna().sum()
df['season_points'] = df['season_points'].fillna(0)
df.isna().sum()

raceId                     0
driverId                   0
constructorId              0
team_name                  0
qualifying_position    16265
pit_stop_count             0
avg_pit_duration           0
season_points              0
year                       0
circuitId                  0
scored_points              0
dtype: int64

From the Kaggle dataset forum:

'qualifying.csv' is incomplete. 1950-1993 is missing, and there are 83 races missiong from 1994 and 2022 seasons. The official F1 website has some data for 1950-1982, but only for the pole sitter. Starting with 1983 includes full info on qualifying data.

In [1752]:
df['qualifying_position']
missing_qual = df[df['qualifying_position'].isna()]
missing_qual[['raceId', 'year', 'driverId', 'team_name', 'qualifying_position']]

# Filter to more recent F1 faces (2002-present) 
df = df[df['year'] >= 2010]

# Fill in remaining NaN in qualifying_position with worst-case (max + 1)
# Note: number of drivers on the grid can change from year to year, hence this loop
worst_positions = df.groupby('raceId')['qualifying_position'].max() + 1
for race_id, worst_pos in worst_positions.items():
    df.loc[(df['raceId'] == race_id) & (df['qualifying_position'].isna()), 'qualifying_position'] = worst_pos

print(df.shape)
print(df.isna().sum())

(6436, 11)
raceId                 0
driverId               0
constructorId          0
team_name              0
qualifying_position    0
pit_stop_count         0
avg_pit_duration       0
season_points          0
year                   0
circuitId              0
scored_points          0
dtype: int64


Note: F1 change the rules in 2010 so that the top 10 drivers would be in the point-scoring window. Prior to that, it was the top 8 drivers. Thus, it makes sense to drop the years before 2010 anyway to maintain consistency in our data.

In [1753]:
# Randomly sample 2000 rows (cap for project)
df = df.sample(n=2000, random_state=42).reset_index(drop=True)
print(df.shape)

(2000, 11)


After merging the datasets and cleaning the data, we plan to use these features to build our model from the dataframe:

- **qualifying_position** : position where the driver started the race, since starting near the front strongly correlates with finishing in the points
- **pit_stop_count** : total number of pit stops made by driver during the race, so pit strategy could influence race results
- **avg_pit_duration** : average time (in milliseconds) of the driver's pit stops. Faster pit stops could give drivers a competitive advantage
- **season_points** : total number of points the driver had prior to the race. Strong, consistent performance in the past could predict future race success
- **year** : year of the race, helps account for performance trends over time (eg. eras of specific team or driver dominance)
- **team_name** : the constructor (team) associated with driver. since Formula 1 is half racing half engineering, the car itself could have a significant effect on point-scoring success
- **circuitId** : track where the race was held. Some teams and drivers perform better on specific circuits (eg. Monoaco circuit requires cars to run with higher levels of downforce due to the track's tight and twisty layout)


In [1754]:
# Get features and target columns

# apply one-hot encoding for 'team_name' and 'circuitId' categorical data
df = pd.get_dummies(df, columns=['team_name', 'circuitId'], drop_first=True)

# define features, split dataset to get feature and target variables
features = ['qualifying_position', 'pit_stop_count', 'avg_pit_duration', 'season_points', 'year']
target = 'scored_points'

encoded_features = []
for col in df.columns:
    # check if column is one-hot encoded team_name or circuitId column
    if col.startswith('team_name_') or col.startswith('circuitId_'):
        encoded_features.append(col)

features += encoded_features

In [1755]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1600, 61)
(400, 61)
(1600,)
(400,)


In [1756]:
# use StandardScaler to scale the numeric features
scaler = StandardScaler()

numeric_features = ['qualifying_position', 'pit_stop_count', 'avg_pit_duration', 'season_points', 'year']

X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])
X_train.head()

,qualifying_position,pit_stop_count,avg_pit_duration,season_points,year,team_name_AlphaTauri,team_name_Alpine F1 Team,team_name_Aston Martin,team_name_Caterham,team_name_Ferrari,...,circuitId_69,circuitId_70,circuitId_71,circuitId_73,circuitId_75,circuitId_76,circuitId_77,circuitId_78,circuitId_79,circuitId_80
968,-0.510181,-0.640796,-0.180676,0.046210,0.023350,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
240,-1.156748,0.224414,-0.183419,-0.141665,-0.647790,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
819,-1.641673,1.954832,-0.201511,0.220665,-1.318929,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
692,0.298027,1.954832,-0.195892,-0.678450,-1.318929,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
420,0.459668,-0.640796,-0.167605,-0.678450,0.470776,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Logistic Regression (baseline)

In [1757]:
model = LogisticRegression(random_state=42).fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'Accuracy Score: {accuracy_score(y_test, y_pred)}')
print(classification_report(y_test, y_pred))

Accuracy Score: 0.7825
              precision    recall  f1-score   support

           0       0.82      0.76      0.79       214
           1       0.74      0.81      0.78       186

    accuracy                           0.78       400
   macro avg       0.78      0.78      0.78       400
weighted avg       0.79      0.78      0.78       400



In [1758]:
# nonlinear svc model with 'rbf' kerlnel doesn't improve results much
model = SVC(kernel='rbf', random_state=42).fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Accuracy Score: {accuracy_score(y_test, y_pred)}')
print(classification_report(y_test, y_pred))

Accuracy Score: 0.78
              precision    recall  f1-score   support

           0       0.81      0.77      0.79       214
           1       0.75      0.80      0.77       186

    accuracy                           0.78       400
   macro avg       0.78      0.78      0.78       400
weighted avg       0.78      0.78      0.78       400



In [1759]:
# random forest classifier
# builds multiple decision trees, combines their predictions
model = RandomForestClassifier(random_state=42).fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Accuracy Score: {accuracy_score(y_test, y_pred)}')
print(classification_report(y_test, y_pred))

Accuracy Score: 0.7725
              precision    recall  f1-score   support

           0       0.81      0.75      0.78       214
           1       0.73      0.80      0.77       186

    accuracy                           0.77       400
   macro avg       0.77      0.77      0.77       400
weighted avg       0.78      0.77      0.77       400



#### Interpretation of Results

All three models performed similarly (acc=77-78%), which suggests the dataset was well-structured and the features were predictive. Logistic regression was a good baseline model to use since it's easily interpretable. SVM performed about the same as the LogisticRegression model. Random Forest have us a slightly lower accuracy than the other 2 models by less than 1% (basically the same).

The feature_importances_ attriubute in the sklearn random forest models gives us a measure of importance of each feature of our dataset in making predictions. It's computed as the mean and standard deviation of accumulation of the impurity decrease within each tree.

In [1760]:
# feature importance for random forest
feature_importance = pd.Series(model.feature_importances_, index=X_train.columns)
feature_importance.head().sort_values(ascending=False)

qualifying_position    0.241100
season_points          0.237272
avg_pit_duration       0.091075
year                   0.066182
pit_stop_count         0.045901
dtype: float64

The feature imporance indicates the qualifying position (0.241) and season points (0.237) are the strongest predictors of whether a driver scores points in a race. This makes sense since drivers who start near the front and have performed well throughout the season are most likely to finish in the top 10, and thus be in the point-scoring range. Average pit duration (0.091) and year (0.066) have additional predictive value. Pit stop count (0.046) has the least influence but is still in the top 5 predictive features, which suggests that pit strategy in racing matters. Contrary to our expectations, the constructor (team) didn't have as much of an effect on point-scoring as we predicted. In conclusion, driver performance and starting position remain the dominant factors in determining race outcomes.

#### Future Improvements

To improve the model's results, we could:

1. Create additional features such as driver experience (number of prior races) or constructor standings to give the model more context. 
2. Refine the dataset by filtering outliers (eg. races with unusual weather conditions) or focus on more recent seasons to reduce noise.
3. Use k-fold cross validation to ensure that the model generalizes well across different subsets of data.
4. Adjust some of the model parameters (eg. C for Logistic Regression, n_estimators and max_depth for Random Forest).

These changes could allow for better feature representation and improved prediction accuracy beyond the results achieved in this project.

### Conclusion

We were able to successfully build machine learning models to predict whether a Formula 1 driver would score points in a race depending on several factors. Logistic Regression, SVM, and Random Forest gave us similar results (77-78% accuracy). An analysis of feature importance confirmed that qualifying position and season points are the most significant predictors. 

Our findings generally aligned with our expectations. Drivers who start near the front and perform consistently throughout the season are the most likely to finish in the points. Future improvements could include adding new features, tuning model parameters, or using cross-validation to further optimize model performance.

### Citations

“Formula One.” Wikipedia, Wikimedia Foundation, 18 Feb. 2019, en.wikipedia.org/wiki/Formula_One.

“History of Formula One.” Wikipedia, Wikimedia Foundation, 14 Oct. 2019, en.wikipedia.org/wiki/History_of_Formula_One.

“List of Formula One World Championship Points Scoring Systems.” Wikipedia, Wikimedia Foundation, 21 May 2024, en.wikipedia.org/wiki/List_of_Formula_One_World_Championship_points_scoring_systems.

Poindexter, Owen. “Formula 1 Unveils Record 24-Race Schedule, Includes Vegas.” Front Office Sports, 21 Sept. 2022, frontofficesports.com/formula-1-unveils-record-24-race-schedule-including-vegas/. Accessed 31 July 2025.